# ACA 2 · Computación en la Nube · RAG integrado

**Integrantes:** Brayan Guillermo Moreno Castro · José Daniel Vargas Otalvaro

**Pilar: ventas.** Sistema conversacional sobre tres documentos académicos
simulados de Superstore. Se integran implementación, interfaz, evaluación
y materiales de entrega de las dos personas. El ACA 1 original se conserva.
El paquete embebido contiene las fuentes originales y los módulos para
regenerar los archivos canónicos; omite copias de resultados derivables
para mantener el cuaderno ágil y autocontenido.

**Estado real de esta versión:** la prueba independiente de Google Colab AI
funcionó y el 15 de noviembre de 2026 se publicó y auditó en Qdrant Cloud
**una colección propia de 47 vectores y siete índices de metadatos**.
Las 13 consultas reales de recuperación evaluaron 12 casos; en 9 se
encontraron todas las secciones esperadas. **Aún no se ha certificado
la respuesta RAG completa con LLM ni su fidelidad**, que requieren
ejecución en el cuaderno con credenciales de acceso habilitadas.
La antigua base del compañero no respondió correctamente y no se toca.
La colección nueva se reconstruyó desde el corpus local del ACA 1,
no de un snapshot del servicio antiguo.
Ver `resultados/publicacion_cloud_propia_20261115.json`,
`resultados/recuperacion_cloud_real_20261115.json` y
`resultados/procedencia_reconstruccion_aca1.json`.

Ejecutar en orden. La preparación del corpus es local. La sección 4B puede
verificar sin cambios una colección coincidente; no activar una escritura
sin confirmar que el destino es el clúster propio.
La aplicación y las demás comprobaciones son de solo lectura en Qdrant.
La demostración usa una llamada LLM si la recuperación está
verificada. La batería de 12 casos / 13 turnos está desactivada hasta
activar su confirmación explícita. Nunca se borra ni se recrea la colección del compañero.


## 1. Materializar el paquete reproducible

El código, corpus, documentos y referencia saneada del ACA 1 están
incluidos dentro del notebook. Se verifica SHA-256 antes de extraer.
La carpeta `ACA2_RAG_Integrado_<version>` identifica el paquete por su hash:
una nueva versión se prepara sin reemplazar archivos ni resultados anteriores.
Si se modificó una carpeta de esa misma versión, se conserva y se informa
el conflicto para elegir otra carpeta de trabajo explícitamente.


In [18]:
# @title Texto de título predeterminado
from pathlib import Path
import ast, base64, hashlib, io, json, os, re, stat, sys, zipfile
from datetime import datetime, timezone

PAQUETE_B64 = '...'
PAQUETE_SHA256 = 'dd26935d9870c81fd76f798754d9f5f8a4617d667c35c0f6a99f68ea80b1cc90'
BASE = Path.cwd() / ("ACA2_RAG_Integrado_" + PAQUETE_SHA256[:12])
MANIFIESTO_EMBEBIDO = {'.env.example': '4b7b0dc6cd9a20b0ed18de78b14d9e509d18c855cb89fdb2a04789de82ccc36c', '.gitignore': '8ffae570c5c692429c28f540ea736f0ffb755d0d70b11cf149f445352949a05d', '.streamlit/config.toml': 'd9b6ae863e66cee502b5f3f65f409171397f2203443781436ab390c84e5e4066', 'README.md': '25b9ef22b2d0e19761dff5159f1c173a4943e3c93aa92d06fc1a767af0fa4907', 'app.py': '597b91688341cf3377e8bb56c645b4c849f34a71e98f959323c7e95aea8d8984', 'banco_pruebas.py': 'e8cd2a2422ffd8f5977a332f0420fcd2dd51b1830a3f97a17b1c0a176bde5979', 'config_proyecto.json': '556bc8c5dccd1ef20c605d3baccd928cd00924cb6c72fc681d74be857d4f1025', 'contenidos_p2.py': '8b0e699f7d41df79e27620797e8450e8908f91470d8151e051bd06a581353b9d', 'corpus/Catalogo_Soporte_Tecnico_Productos.md': '49cefd7fd0fddb8a6dff5cf7fca9c5b395e02ebfd99dece0338bf0f1e51e0717', 'corpus/Manual_SLA_Logistica_Envios.md': 'ceb2969d44d0de5b989297defd0bd84cf75de02fd927b0e2ff5e226a57aee14c', 'corpus/Politica_Devoluciones_Garantias_Superstore.md': '69396a2b55d97c8c8ff04f05e5c5015c575698b3687514a9717b9e1d48b884f0', 'corpus/catalogo_metadatos.json': 'b88527faf1d9ed15f077fb642fd2e9f469202b7014f29a20dc39d947b2d631d9', 'corpus_fuente.json': '1e34a469278740a59752a91c75d39308b48c7cf67e864d39923e2ae3a3f32ef2', 'datos/corrida_01.json': 'e14433e64f5055a1234b538bad80adade915f651422afe614434bcfc8b24dcff', 'documentacion/banco_pruebas.md': '01e0b45cc7267325363332f11e2fd5d1ed083511bb0892c33e2471caa073ea5e', 'documentacion/checklist_entrega.md': 'd968716eb63a9acf7b4dbce48958a720f26947e575e1a096c4f8b627ae195e9c', 'documentacion/estado_integracion.md': 'c8b2fee12b55722df29a75647c373006cc617d4364bcd7e8586aaefaefe8abbf', 'documentacion/guion_video_4m30.md': '1b1145338c0e8e606442bf5f8ed4bca736d8546bf20dc23fbda9cd68b113e69f', 'documentacion/handoff_persona1.md': '08c9bc14c3436e75aecc4adf811ab97c7b4851349b04a68a0523de61b2e4bca4', 'documentacion/informe_integrante2.md': '6d685e0ad45e2de7cfea5f026c380b64723f4a0405b445884ef38097a54af6c5', 'documentacion/manual_verificacion.md': 'ede9b854de7a07cd4248872c32838fc1e734044180fd52dc380f24d1a1cf699d', 'documentacion/matriz_rubrica.md': '19501a053150376353c94cfe3cfb94931baa86cdd08c40e18b60bd53da874d75', 'documentacion/referencias_APA7.md': '77d327f0bebbb034d07e1efc4c9143860c2c39604e8e8b9d0c153d477c0ed14f', 'evaluacion_aca2.py': '5edc63ed978bda5c2c177d1fe93b285163bcd1169d08e8b9ecdc4f5eaccf3d47', 'generar_entrega.py': '1003672358ac891ce3c98662c51e55c9ed0c7a2532c66322725cff42774bfe9e', 'logo_cun.png': '85e0d9cc2789e6e6b17d86a7f3ea954eb7c7eb0da8bebea530584a82da78841a', 'materiales/Documentacion_ACA2_Integrado.pdf': '0392c00bccbdff5b3eb87cc8e09ae847adef2a9fe669c21baffdd7c22d567c3c', 'materiales/Guion_video_4m30.pdf': 'f093a8f29c94be6db05c97eb6e7fbd6aa89bf58adb64e1e6f67a856706129012', 'materiales/Poster_ACA2_BORRADOR.pdf': '60c6ff55ff08ead754ff21ec44827468a5e218e2b5352abf58d5dbd0791f2b17', 'materiales/Poster_editable.html': 'c9ce1d76e15b60c5297996e548c9b430c6d7bd2fd3af154d684f63d0f115a6da', 'materiales_aca2.py': '925ee6c0f495cc27b7ffebff4f061b0cc5a8db21d325915aaffcc3b7c0165f24', 'p1_adapter.py': 'c6b1c6bb179878ef917c5331d826871bf422ac9441c3ed519ba25fe52b2a50cc', 'rag_backend.py': '6b7f00c3e77e10d3c1a7954eee34c9033dc0f78fe5fe5569d0a7554d03551e39', 'reconstruir_qdrant.py': 'fb6739bd7e7e0e2dcaddceea2f737620f701f2ff11f8c8ca3e7e7b66184c4d27', 'referencia/ACA1_base_sanitizada.ipynb': '152bb27060c11ecdf37b43f2614dd28204c60e7f5defd164b38a65b7dcf4d733', 'referencia/guias/ACA_2_Computacion_Nube_RAG_CUN.pdf': 'dfa9778612b580482e672d961ebbb8bd06e6b9b9441966c18e0d4b0a2c4eb9e1', 'referencia/guias/Distribucion_ACA_2_Computacion_Nube_RAG.pdf': 'd11326d576e546c2eb9cfe9acf21f05ceafd59acf964c096011db31bb8d5d756', 'referencia/recursos_aca1/README.md': '4503f7953ae33a8783c701e0dfc1151fb1bece63c7cb642b8d2a22265ab32102', 'referencia/recursos_aca1/aca1_pipeline.py': '0dfb7ae12175c619e3b43aa66058f2e9c085e92499bbe4d3c56aafb451422ff9', 'referencia/recursos_aca1/corpus_content.py': '6c6d7f2a82c0c9be222531cb1bb355585f9aaa87dea72cb67319e02c60338ae4', 'referencia/recursos_aca1/dataset_original.zip': 'a9b35af6cf9530e61d42ec22ef6d485799c0ee7d2e4710f1f803a71b8f1c29cb', 'referencia/recursos_aca1/integrar_notebooks.py': '60729aea5393efca99c6dba8233098c8ad55e7f833c115546d24864e160b0325', 'referencia/recursos_aca1/materiales_p1.py': '4790a2d07f281d43cd6a99b05b26be360d70714981c78c7641c36a0864afaef0', 'referencia/recursos_aca1/requirements.txt': 'bf1c1a5d2ed466e77f3fb9e6e8dfa33708b30ecd8a5c3e4865a7c98de2652b14', 'requirements.txt': '1968d51fa4a8c2ec5c6878f02c0e762328071a8ead59a64542ddb929396e9ff3', 'resultados/autopruebas_base_propia.json': 'ee05afc34006165c1f09365081a2046febb13a280211322b1580efdd5996dd7e', 'resultados/autopruebas_integradas.json': '51ef8544a216149340a1c5b4d9380df245c2349cc4befddae7185929f15af497', 'resultados/comprobacion_cierre_20261115.json': '2506435513fb28644e3f8bffa20c6903b9be17ae589d0cfde56eb60c098f7c8c', 'resultados/embeddings_reconstruccion_local.json': '984dcec5519c306340c986b0677dd2afe293ce2e21bb1a01f42d8397c8c7fab6', 'resultados/incidencias.json': '4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba873c2f11161202b945', 'resultados/metricas.json': 'd80677947a26ba967ba354166a55472dec7cdab8e9e2ac20127760dc6bff7e1a', 'resultados/preparacion_local_base_propia.json': '0a0b4a8c7eb6c2b10be9af3a5a890d9630d3a8f40ae8e58735ad0af0c7f9974b', 'resultados/procedencia_reconstruccion_aca1.json': '3398344a65ab057bd10873ed965093814b72d537bf103eb75a7cbafbcf11f572', 'resultados/publicacion_cloud_propia_20261115.json': '8ffc26c00216be882649a64de7f1b2e922c4db9a59d9db542d40e95736ae8b9e', 'resultados/recuperacion_cloud_real_20261115.json': 'cecd7d2e1bcda5348cdc6eacdee3a4d80f104b85328a67206a44408131ff9a84', 'resultados/recuperacion_local_reconstruida.json': '6e4dd026d5a63785bf589e14285f7b6dc4309db24953a56cf468557a055291c2', 'resultados/registro_evaluacion.csv': '1de0c2348a260d3578eacf9cdc7d555a8fb1cd04e38a218664d36766f1365e93', 'resultados/resultados_reales.json': 'e14433e64f5055a1234b538bad80adade915f651422afe614434bcfc8b24dcff', 'resultados/salidas_colab_pre_cierre_20261115.json': 'd03b25e4e31861d0a4c8767e515f78b45bb69d7884f5a6aca22d46ca6230dd01', 'resultados/verificacion_servicios_20261115.json': '1d8c5de53375073873d3a657b0aa10f546ee26a9a0e89cc0b0467e570aeaa406', 'rubrica_p2.py': '75a35f2ab4a587e45f5777024cf040a40be5cee60343bd8a60266a961b180cd9', 'test_app_contract.py': 'b0e616012bf9e780417ae5230a5462898669267855f23c3954e03c5fc3ed4414', 'test_evaluacion_aca2.py': '664052c8bc28a4e6461776da2a106de4b5fa184c5dad8d3916e20a330e645814', 'test_rag_backend.py': 'e6693135a7d1a7f9cba020645c36a165dbe67dd0ffa94490f9513bdbc02872f4', 'test_reconstruir_qdrant.py': '935a70c4c58c30e2677263a306178d609a298b2ca4f5ea99ebaf2bf0cfcfd2a9'}
raw = base64.b64decode(PAQUETE_B64, validate=True)
if hashlib.sha256(raw).hexdigest() != PAQUETE_SHA256:
    raise ValueError("Integridad del paquete no verificada. No se extrajo ningún archivo.")
BASE.mkdir(parents=True, exist_ok=True)
BASE = BASE.resolve()
nuevos, conflictos = [], []
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for item in z.infolist():
        rel = Path(item.filename)
        if item.is_dir() or rel.is_absolute() or ".." in rel.parts:
            raise ValueError("Ruta no admisible en el paquete.")
        if stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError("No se permiten enlaces simbólicos en el paquete.")
        dest = (BASE / rel).resolve()
        if BASE not in dest.parents:
            raise ValueError("Ruta fuera de la carpeta autorizada.")
        contenido = z.read(item)
        esperado = MANIFIESTO_EMBEBIDO.get(rel.as_posix())
        if hashlib.sha256(contenido).hexdigest() != esperado:
            raise ValueError("No coincide el hash de un archivo del paquete.")
        if dest.exists():
            if not dest.is_file() or hashlib.sha256(dest.read_bytes()).hexdigest() != esperado:
                conflictos.append(rel.as_posix())
        else:
            nuevos.append((dest, contenido))
    if conflictos:
        raise ValueError("Se conservaron archivos modificados. Use una carpeta nueva: " + ", ".join(conflictos[:10]))
    for dest, contenido in nuevos:
        dest.parent.mkdir(parents=True, exist_ok=True)
        with dest.open("xb") as f:
            f.write(contenido)
del raw, PAQUETE_B64, nuevos
print("Paquete verificado:", len(MANIFIESTO_EMBEBIDO), "archivos; carpeta:", BASE.name)


Paquete verificado: 65 archivos; carpeta: ACA2_RAG_Integrado_dd26935d9870


## 2. Entorno y dependencias

Instalar las versiones especificadas en `requirements.txt`. Esta operación
descarga dependencias de sus canales habituales; el paquete del proyecto
no depende de una URL de descarga externa. Los pesos del mismo modelo de
embeddings del ACA 1 se cargan cuando se haga una consulta semántica.
No se instala otro LLM local ni se consume GPU para generación local.


In [19]:
import subprocess
INSTALAR_DEPENDENCIAS = False  # @param {type:"boolean"}
if INSTALAR_DEPENDENCIAS:
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(BASE / "requirements.txt")],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    if install.returncode:
        raise RuntimeError("Instalación no completada. Revisar el entorno de forma privada; no se imprimieron logs.")
    print("Dependencias instaladas; no se imprimieron datos de configuración.")
    del install
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
# No reutilizar por accidente módulos homónimos de otra carpeta.
MODULOS_PROYECTO = ["app", "p1_adapter", "rag_backend", "materiales_aca2", "contenidos_p2",
                   "rubrica_p2", "evaluacion_aca2", "banco_pruebas", "test_evaluacion_aca2",
                   "test_rag_backend", "test_app_contract", "reconstruir_qdrant", "test_reconstruir_qdrant"]
for nombre in MODULOS_PROYECTO:
    mod = sys.modules.get(nombre)
    if mod is not None and getattr(mod, "__file__", None):
        if Path(mod.__file__).resolve().parent != BASE:
            del sys.modules[nombre]
from rag_backend import Config, RagService, RagError, APP_VERSION, PROMPT_VERSION, EMBEDDING_MODEL
from banco_pruebas import CASOS, FUNCIONALES
from evaluacion_aca2 import (cargar_json, guardar_json, catalogar_corpus, validar_banco,
    inicializar_registro, ejecutar_bateria, revisar_caso, registrar_funcional,
    calcular_metricas, pendientes_cierre, exportar_registros, validar_coherencia_exportacion)
from rubrica_p2 import RUBRICA, ENTREGABLES
from IPython.display import display, Markdown, FileLink
CONFIG = cargar_json(BASE / "config_proyecto.json")
CORPUS = cargar_json(BASE / "corpus_fuente.json")
CATALOGO = catalogar_corpus(CORPUS)
resumen_banco = validar_banco(CASOS, FUNCIONALES, CORPUS)
print(resumen_banco)
# Esta evidencia histórica no se reemplaza por el resultado de una prueba nueva.
display(cargar_json(BASE / "resultados/verificacion_servicios_20261115.json"))

def guardar_evidencia_nueva(nombre, datos):
    """Registro nuevo e inmutable por intento; nunca reemplaza uno anterior."""
    from uuid import uuid4
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    ruta = BASE / "resultados" / "verificaciones" / (nombre + "_" + stamp + "_" + uuid4().hex[:8] + ".json")
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with ruta.open("x", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2, allow_nan=False)
    return ruta


{'casos_conversacionales': 12, 'turnos_previstos': 13, 'pruebas_funcionales': 4, 'documentos_locales': 3, 'secciones_documentales': 16, 'integridad_banco': 'verificada_localmente', 'nota': 'Secciones no equivalen a fragmentos/chunks. No se verificó el índice cloud ni el LLM.'}


{'fecha_local': '2026-11-15',
 'zona_horaria': 'America/Bogota',
 'tipo': 'observaciones_reales_de_conectividad_no_evaluacion_de_fidelidad',
 'colab_destino_id': '1oCnWKoHeu2l8NO7vB66yo3SLw_d7XNzc',
 'llm': {'proveedor': 'Google Colab AI',
  'modelo_solicitado': 'google/gemini-3.8-flash',
  'api_publica': 'google.colab.ai.generate_text',
  'prueba_desde_celda': {'respuesta': 'CONEXION_LLM_OK', 'real': True},
  'prueba_desde_subproceso': {'codigo_salida': 0,
   'respuesta': 'APP_LLM_OK',
   'real': True},
  'requiere_clave_gemini_en_esta_ruta': False,
  'version_resuelta_expuesta': False,
  'parametros_temperatura_y_tokens_expuestos': False,
  'alcance': 'Las dos llamadas verificaron acceso al modelo. No consultaron Qdrant ni prueban fidelidad RAG.'},
 'qdrant': {'coleccion_objetivo': 'superstore_rag_docs',
  'dns_resuelve_en_colab': True,
  'colab_puerto_6333': 'ConnectionResetError antes de respuesta HTTP',
  'colab_puerto_443': 'ConnectionResetError antes de respuesta HTTP',
  'compr

## 3. Problema, objetivo y arquitectura

Una búsqueda de palabras puede omitir consultas equivalentes y no construye
una respuesta conversacional fundamentada. El objetivo es integrar el índice
semántico del ACA 1 con un LLM real y una UI que muestre de dónde sale cada
respuesta. RAG no entrena el modelo ni convierte los documentos ficticios en
políticas reales.

1. La pregunta, con la última referencia del usuario si hay seguimiento,
   se representa con `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensiones).
2. Qdrant recupera Top-k fragmentos de `superstore_rag_docs_aca2_propio`, por
   coseno, una vez publicada y verificada la reconstrucción. La colección
   histórica `superstore_rag_docs` no se modifica.
3. El prompt separa instrucciones, memoria y documentos con etiquetas `[F1]`.
   La memoria ayuda a interpretar referencias; no aporta hechos verificables.
4. Gemini genera una respuesta y sus citas. El backend valida su estructura y
   bloquea referencias desconocidas; una revisión posterior evalúa fidelidad.
5. Streamlit mantiene historial por sesión y permite explorar documentos,
   metadatos, índice y fragmentos. No incluye altas/borrados del índice.

El prompt exige abstenerse sin evidencia, pedir aclaración ante ambigüedad,
ignorar instrucciones incrustadas en documentos y señalar la simulación.
La similitud y la existencia de una cita **no prueban por sí solas** que una
afirmación esté respaldada. Los controles reducen, no eliminan, alucinaciones.


In [20]:
display([{ "criterio": r["criterio"], "peso_porcentaje": r["peso"],
           "evidencia_requerida": r["evidencia"] } for r in RUBRICA])
print("Entregables oficiales:")
for item in ENTREGABLES:
    print("-", item)
display([{ "fuente": d["filename"], "secciones": len(d["sections"]),
           "naturaleza": "académica simulada" } for d in CORPUS["DOCS"]])
corpus_sha = hashlib.sha256((BASE / "corpus_fuente.json").read_bytes()).hexdigest()
if CONFIG.get("corpus_sha256") != corpus_sha:
    raise ValueError("El corpus local difiere de la versión documentada. No continuar ni mezclar evaluaciones.")
print("Corpus local comprobado: 3 documentos / 16 secciones; no equivale al número de chunks remotos.")


[{'criterio': 'Integración RAG',
  'peso_porcentaje': 20,
  'evidencia_requerida': 'Ejecución real y reproducible con pregunta, fragmentos Top-k, identificación de fuentes, contexto suministrado al LLM y respuesta generada. Registrar versión del sistema y del corpus.'},
 {'criterio': 'Configuración LLM',
  'peso_porcentaje': 20,
  'evidencia_requerida': 'Identificación del modelo y modalidad, parámetros documentados, justificación de la elección, llamada real al LLM y revisión de código y salidas sin divulgar valores de credenciales.'},
 {'criterio': 'Interfaz Gráfica (UI)',
  'peso_porcentaje': 20,
  'evidencia_requerida': 'Recorrido real del chat y de un seguimiento conversacional; capturas de respuestas con fuentes y del panel documental; comprobación de acceso a la aplicación.'},
 {'criterio': 'Fidelidad de Respuestas',
  'peso_porcentaje': 15,
  'evidencia_requerida': 'Casos prefijados con pregunta, contexto del diálogo, fuente esperada, respuesta obtenida, citas, tiempo y juicio 

Entregables oficiales:
- Notebook Técnico Reproducible (.ipynb)
- Póster Sintético tipo Infografía (.pdf)
- Aplicación Interactiva y Código Fuente (GitHub)
- Video de Sustentación (3 a 5 min)


[{'fuente': 'Politica_Devoluciones_Garantias_Superstore.md',
  'secciones': 5,
  'naturaleza': 'académica simulada'},
 {'fuente': 'Manual_SLA_Logistica_Envios.md',
  'secciones': 6,
  'naturaleza': 'académica simulada'},
 {'fuente': 'Catalogo_Soporte_Tecnico_Productos.md',
  'secciones': 5,
  'naturaleza': 'académica simulada'}]

Corpus local comprobado: 3 documentos / 16 secciones; no equivale al número de chunks remotos.


## 4. Configuración segura y justificación del LLM

La ruta principal es `google.colab.ai` con el modelo disponible y comprobado
en esta cuenta: `google/gemini-3.8-flash`. El SDK gestiona autenticación, sin
copiar tokens internos ni pedir una API key adicional. En esta interfaz el
proveedor administra temperatura y límite de salida; no se presentan como
parámetros controlados ni se inventan consumos de tokens o versión resuelta.
Su disponibilidad depende de la cuenta, modelo y cuota, no es ilimitada.

El destino es **propio**, con `QDRANT_PROPIO_URL` y `QDRANT_PROPIO_API_KEY`
del entorno o de **Secretos de Colab**, con acceso habilitado al notebook.
No se reutilizan `QDRANT_URL` ni `QDRANT_API_KEY` del compañero. La colección
nueva se llama `superstore_rag_docs_aca2_propio`. Solo se muestra si las claves existen.
Si se cambia un secreto durante la sesión, activar
`RECARGAR_SECRETOS_COLAB` para reemplazar únicamente los valores propios
en memoria; un fallo de lectura no conserva una credencial anterior.
No pegarlos en celdas, chat, capturas o repositorio. Fuera de Colab existe
la alternativa `RAG_LLM_BACKEND=gemini_api`, con un modelo REST válido y
`GEMINI_API_KEY` propia; no se declara evaluada automáticamente.

`corpus_version=1.0.0` identifica la edición documental, no el hash de versión
que guarda Qdrant. No usarlo como filtro remoto. Se verificará la versión
efectivamente recuperada antes de fijar un filtro.


In [21]:
os.environ.setdefault("RAG_LLM_BACKEND", "colab_ai")
os.environ.setdefault("GEMINI_MODEL", "google/gemini-3.8-flash" if os.environ["RAG_LLM_BACKEND"] == "colab_ai" else "gemini-2.5-flash-lite")
os.environ["RAG_QDRANT_DESTINO"] = "propio"
# El endpoint es PÚBLICO, no una credencial. Simplifica la configuración:
# solo QDRANT_PROPIO_API_KEY debe guardarse como secreto privado de Colab.
os.environ.setdefault("QDRANT_PROPIO_URL", "https://840d160f-288c-4d44-a869-96d057e423d3.sa-east-1-0.aws.cloud.qdrant.io")
os.environ.pop("RAG_CORPUS_VERSION", None)  # Auditar de nuevo el destino propio.
os.environ.pop("RAG_EMBEDDING_REVISION", None)  # Se fija desde el manifiesto verificado en 4A.
# Cambiar/releer configuración invalida la evidencia derivada de otra sesión.
INDICE_OK = CORPUS_NUBE_OK = RECUPERACION_OK = DEMO_REAL_OK = False
BASE_PROPIA_PUBLICADA = False
ESTADO_INDICE, FRAGMENTOS_NUBE = None, []
CONFIG.update(qdrant_conexion_real_verificada=False, llm_conexion_real_verificada=False,
              corpus_cloud_verificado=False, base_propia_publicada=False,
              corpus_version_remota_verificada=None, filtro_version_remota_activado=False,
              reconstruccion_manifest_sha256=None, modelo_embeddings_revision=None,
              estado_integracion="Configuración propia renovada; comprobaciones reales pendientes en esta sesión.")
requeridos = ["QDRANT_PROPIO_URL", "QDRANT_PROPIO_API_KEY"]
RECARGAR_SECRETOS_COLAB = False  # @param {type:"boolean"}
if os.environ["RAG_LLM_BACKEND"] == "gemini_api":
    requeridos.append("GEMINI_API_KEY")
for nombre in requeridos:
    if RECARGAR_SECRETOS_COLAB:
        os.environ.pop(nombre, None)
    if not os.environ.get(nombre):
        try:
            from google.colab import userdata
            valor = userdata.get(nombre)
            if valor:
                os.environ[nombre] = valor
            del valor
        except Exception:
            pass  # No se imprime la excepción ni el valor del secreto.
# Alternativa únicamente para la sesión actual si Secretos de Colab
# no permite crear una entrada: getpass oculta el valor, no lo guarda
# en el notebook ni en archivos. Se pierde al desconectar el runtime.
USAR_CLAVE_EFIMERA = False  # @param {type:"boolean"}
if USAR_CLAVE_EFIMERA and not os.environ.get("QDRANT_PROPIO_API_KEY"):
    from getpass import getpass
    clave_temporal = getpass("Clave privada de Qdrant para ESTA sesión (entrada oculta): ")
    if clave_temporal:
        os.environ["QDRANT_PROPIO_API_KEY"] = clave_temporal
    del clave_temporal
servicio, cfg, configuracion_ok = None, None, False
try:
    cfg = Config.from_env()
    servicio = RagService(cfg)
    public = cfg.public_dict()
    CONFIG.update(modelo_llm=public["modelo_llm"], proveedor_llm=public["proveedor_llm"],
                  llm_backend=public["llm_backend"], temperatura=public["temperatura"],
                  max_output_tokens=public["max_output_tokens"], version_app=APP_VERSION,
                  version_prompt=PROMPT_VERSION, destino_qdrant=public["destino_qdrant"],
                  coleccion_activa=public["coleccion"], usa_credenciales_heredadas=False)
    configuracion_ok = bool(cfg.qdrant_url and cfg.qdrant_api_key)
    display(public)
    if not configuracion_ok:
        print("Falta la conexión propia. La preparación local 4A puede ejecutarse; no se intentó usar el clúster del compañero.")
except RagError as error:
    print("Configuración bloqueada:", str(error))
except Exception:
    print("Configuración bloqueada. Revisar valores de forma privada; no se mostraron secretos.")


{'coleccion': 'superstore_rag_docs_aca2_propio',
 'destino_qdrant': 'propio',
 'modelo_llm': 'google/gemini-3.8-flash',
 'proveedor_llm': 'Google Colab AI',
 'llm_backend': 'colab_ai',
 'requiere_gemini_key': False,
 'modelo_embeddings': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
 'modelo_embeddings_revision': None,
 'dimension': 384,
 'temperatura': None,
 'max_output_tokens': None,
 'parametros_gestionados_por_proveedor': True,
 'timeout_qdrant_s': 60.0,
 'timeout_llm_s': None,
 'version_app': '2.1.0-base-propia',
 'version_prompt': 'rag-ventas-v2.0.0',
 'corpus_version_filtro': None,
 'claves_configuradas': {'QDRANT_PROPIO_URL': True,
  'QDRANT_PROPIO_API_KEY': True,
  'GEMINI_API_KEY': False},
 'solo_lectura_qdrant': True}

## 4A. Reconstruir el corpus del ACA 1 en una carpeta propia

Esta preparación **no se conecta a Qdrant**. Reproduce los documentos,
metadatos y fragmentos desde `corpus_fuente.json` y el pipeline saneado del
ACA 1. Los Markdown canónicos van en una carpeta separada: no se sobrescriben
las fuentes de presentación ni se reutilizan celdas destructivas del original.

Se mantienen fragmentación 500/80, MiniLM multilingüe, 384 dimensiones,
normalización de vectores y distancia Cosine. Antes de generar embeddings se
mide la longitud con el tokenizer real; no se permite truncamiento silencioso.
`PREPARAR_EMBEDDINGS=True` descarga el modelo desde su fuente habitual si
no está en caché. No invoca al LLM ni consume la cuota de Qdrant.

La igualdad comprobada corresponde al corpus local de ACA 1. Sin snapshot
ni acceso a la base anterior **no se afirma que sea una copia idéntica del
índice remoto**. La preparación local tampoco acredita publicación cloud.


In [22]:
from reconstruir_qdrant import (preparar_corpus, preparar_embeddings,
                               publicar_reconstruccion, DestinoPropio, ReconstruccionError)
PREPARAR_EMBEDDINGS = True  # @param {type:"boolean"}
PREPARACION_PROPIA = None
reconstruccion_local = {"operacion": "reconstruccion_local_aca1", "correcto": False,
                        "publicacion_cloud": False}
try:
    PREPARACION_PROPIA = preparar_corpus(BASE)
    display(PREPARACION_PROPIA.informe())
    if PREPARAR_EMBEDDINGS:
        PREPARACION_PROPIA = preparar_embeddings(PREPARACION_PROPIA)
        display(PREPARACION_PROPIA.informe())
    if PREPARACION_PROPIA.publicable:
        revision_embeddings = PREPARACION_PROPIA.manifiesto.get("modelo_revision")
        if not isinstance(revision_embeddings, str) or not re.fullmatch(r"[a-fA-F0-9]{40}", revision_embeddings):
            raise ReconstruccionError("Falta la revisión verificable del modelo: no se autoriza mezclar vectores con una revisión desconocida.")
        os.environ["RAG_EMBEDDING_REVISION"] = revision_embeddings
        CONFIG["modelo_embeddings_revision"] = revision_embeddings
        cfg = Config.from_env()
        servicio = RagService(cfg)
    reconstruccion_local.update(correcto=True, detalle=PREPARACION_PROPIA.informe(),
                                 embeddings_listos=PREPARACION_PROPIA.publicable)
    CONFIG["reconstruccion_manifest_sha256"] = PREPARACION_PROPIA.huella
    print("Fuentes reconstruidas sin consultar ni modificar la base del compañero.")
except ReconstruccionError as error:
    reconstruccion_local["incidencia"] = str(error)
    print("Reconstrucción detenida sin publicar:", str(error))
except Exception as error:
    reconstruccion_local["incidencia"] = "Fallo controlado: " + type(error).__name__
    print("Preparación no completada. Revisar dependencias o auditoría local; no se mostraron datos privados.")
guardar_evidencia_nueva("reconstruccion_local", reconstruccion_local)


{'schema_version': 'aca2-reconstruccion-1.0',
 'estado': 'corpus_preparado',
 'naturaleza': 'reconstruccion_desde_fuentes_locales_no_copia_remota',
 'modelo_embeddings': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
 'dimension': 384,
 'distancia': 'Cosine',
 'chunk_size': 500,
 'chunk_overlap': 80,
 'corpus_version': 'superstore-academico-575b82ca54a0',
 'chunk_count': 47,
 'doc_count': 3,
 'section_count': 16,
 'coleccion_destino': 'superstore_rag_docs_aca2_propio',
 'fuentes': {'corpus_sha256': '1e34a469278740a59752a91c75d39308b48c7cf67e864d39923e2ae3a3f32ef2',
  'dataset_zip_sha256': 'a9b35af6cf9530e61d42ec22ef6d485799c0ee7d2e4710f1f803a71b8f1c29cb',
  'pipeline_sha256': '0dfb7ae12175c619e3b43aa66058f2e9c085e92499bbe4d3c56aafb451422ff9'},
 'fixture': False,
 'manifiesto_sha256': 'a0f1df22e8176903db09475b1c56b26d4988d01cbdec725c544039faa1ef4fcb',
 'publicable': False,
 'carpeta_canonica': 'aca1_canonico_20aa15af95462d64_fuentes',
 'nota': 'Reconstrucción desde fuent

{'schema_version': 'aca2-reconstruccion-1.0',
 'estado': 'embeddings_verificados',
 'naturaleza': 'reconstruccion_desde_fuentes_locales_no_copia_remota',
 'modelo_embeddings': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
 'dimension': 384,
 'distancia': 'Cosine',
 'chunk_size': 500,
 'chunk_overlap': 80,
 'corpus_version': 'superstore-academico-575b82ca54a0',
 'chunk_count': 47,
 'doc_count': 3,
 'section_count': 16,
 'coleccion_destino': 'superstore_rag_docs_aca2_propio',
 'fuentes': {'corpus_sha256': '1e34a469278740a59752a91c75d39308b48c7cf67e864d39923e2ae3a3f32ef2',
  'dataset_zip_sha256': 'a9b35af6cf9530e61d42ec22ef6d485799c0ee7d2e4710f1f803a71b8f1c29cb',
  'pipeline_sha256': '0dfb7ae12175c619e3b43aa66058f2e9c085e92499bbe4d3c56aafb451422ff9'},
 'tokenizacion': {'limite': 128,
  'max_tokens': 108,
  'min_tokens': 35,
  'truncados': 0},
 'modelo_revision': 'e8f8c211226b894fcb81acc59f3b34ba3efd5f42',
 'entorno': {'langchain-text-splitters': '1.0.0',
  'numpy': '2.4.6

Fuentes reconstruidas sin consultar ni modificar la base del compañero.


PosixPath('/content/ACA2_RAG_Integrado_dd26935d9870/resultados/verificaciones/reconstruccion_local_20261115T215414459024Z_3938d0af.json')

## 4B. Crear la colección en un Qdrant Cloud propio

Entrar en [Qdrant Cloud](https://cloud.qdrant.io/) con una cuenta propia,
crear un clúster gratuito y guardar su conexión en los secretos nuevos
`QDRANT_PROPIO_URL` y `QDRANT_PROPIO_API_KEY`. Ejecutar nuevamente la sección 4
si se agregaron después. La clave de base de datos necesita permiso para
crear la colección propia durante esta ingesta; no debe publicarse.

La siguiente celda está desactivada hasta confirmar el destino propio.
Crea `superstore_rag_docs_aca2_propio` con sus puntos y campos indexados,
o verifica una reconstrucción que ya coincide. No vacía, borra ni recrea
colecciones y rechaza el host heredado. Si hay diferencias o una ingesta
parcial, conserva el conflicto para revisión; no lo corrige borrando datos.

El clúster gratuito tiene condiciones de inactividad; no se promete
persistencia indefinida. Véase la
[documentación oficial](https://qdrant.tech/documentation/cloud/create-cluster/).
Para acreditar la entrega se deben completar después los controles 5–7 y
la batería real, no basta con que esta celda termine.


In [23]:
PUBLICAR_BASE_PROPIA = False  # @param {type:"boolean"}
BASE_PROPIA_PUBLICADA = False
CONFIG["base_propia_publicada"] = False
INDICE_OK = CORPUS_NUBE_OK = RECUPERACION_OK = DEMO_REAL_OK = False
CONFIG.update(qdrant_conexion_real_verificada=False, corpus_cloud_verificado=False,
              llm_conexion_real_verificada=False, corpus_version_remota_verificada=None,
              filtro_version_remota_activado=False)
if PUBLICAR_BASE_PROPIA:
    publicacion_propia = {"operacion": "publicar_reconstruccion_cloud_propia", "correcto": False}
    try:
        if PREPARACION_PROPIA is None or not PREPARACION_PROPIA.publicable:
            raise ReconstruccionError("Primero completar la preparación y auditoría real de embeddings en 4A.")
        if CONFIG.get("modelo_embeddings_revision") != PREPARACION_PROPIA.manifiesto.get("modelo_revision"):
            raise ReconstruccionError("Reejecutar 4A: la revisión del modelo no coincide con los vectores preparados.")
        destino_propio = DestinoPropio.from_env()
        resultado_publicacion = publicar_reconstruccion(
            PREPARACION_PROPIA, destino_propio, confirmar_creacion=True)
        BASE_PROPIA_PUBLICADA = (
            not PREPARACION_PROPIA.fixture
            and resultado_publicacion.get("modo") == "reconstruccion_cloud_real"
            and resultado_publicacion.get("puntos_verificados") == len(PREPARACION_PROPIA.chunks)
            and resultado_publicacion.get("accion") in ("creada", "verificada_sin_cambios"))
        publicacion_propia.update(correcto=BASE_PROPIA_PUBLICADA, detalle=resultado_publicacion)
        CONFIG["base_propia_publicada"] = BASE_PROPIA_PUBLICADA
        CONFIG["reconstruccion_manifest_sha256"] = PREPARACION_PROPIA.huella
        cfg = Config.from_env()
        servicio = RagService(cfg)
        configuracion_ok = bool(cfg.qdrant_url and cfg.qdrant_api_key)
        display(resultado_publicacion)
        print("Continuar con índice, corpus y consultas reales para comprobar la integración.")
    except ReconstruccionError as error:
        publicacion_propia["incidencia"] = str(error)
        print("No se completó la publicación:", str(error))
    except Exception as error:
        publicacion_propia["incidencia"] = "Fallo controlado: " + type(error).__name__
        print("Publicación no verificada. No se repitió la escritura ni se mostraron datos de conexión.")
    guardar_evidencia_nueva("publicacion_propia", publicacion_propia)
else:
    print("No se escribió en Qdrant. Falta confirmar la publicación en el clúster propio.")


No se escribió en Qdrant. Falta confirmar la publicación en el clúster propio.


## 5. Conexión de solo lectura y supervisión del índice

Se consultan metadatos de colección, conteo exacto y una muestra de payload.
Se exige 384 dimensiones y coseno. Un error de conectividad se conserva como
incidencia: esta comprobación no escribe ni sustituye la nube por un
resultado local. La reconstrucción se autoriza por separado en 4B.
La UI puede iniciarse aun si Qdrant no está disponible.


In [24]:
INDICE_OK = False
CORPUS_NUBE_OK = RECUPERACION_OK = DEMO_REAL_OK = False
CONFIG.update(corpus_cloud_verificado=False, llm_conexion_real_verificada=False,
              corpus_version_remota_verificada=None, filtro_version_remota_activado=False)
ESTADO_INDICE = None
verificacion_indice = {"operacion": "indice_cloud_solo_lectura", "correcto": False,
                       "fecha_utc": datetime.now(timezone.utc).isoformat()}
if configuracion_ok:
    try:
        ESTADO_INDICE = servicio.estado_indice()
        INDICE_OK = bool(ESTADO_INDICE["compatible"] and ESTADO_INDICE["puntos"] > 0)
        verificacion_indice.update(correcto=INDICE_OK, detalle=ESTADO_INDICE)
        display(ESTADO_INDICE)
    except RagError as error:
        verificacion_indice["incidencia"] = str(error)
        print("RAG bloqueado por dependencia real:", str(error))
    except Exception as error:
        verificacion_indice["incidencia"] = "Fallo controlado: " + type(error).__name__
        print("No se pudo verificar el índice. No se imprimieron datos de conexión.")
else:
    verificacion_indice["incidencia"] = "Configuración incompleta; no se intentó conectar."
guardar_evidencia_nueva("indice", verificacion_indice)
CONFIG["qdrant_conexion_real_verificada"] = INDICE_OK


{'coleccion': 'superstore_rag_docs_aca2_propio',
 'estado': 'green',
 'puntos': 47,
 'indexados': 0,
 'dimension': 384,
 'distancia': 'Cosine',
 'compatible': True,
 'solo_lectura': True,
 'version_corpus': 'superstore-academico-575b82ca54a0',
 'version_corpus_muestra': True,
 'metadatos_indexados': ['metadata.category',
  'metadata.corpus_version',
  'metadata.doc_source',
  'metadata.region',
  'metadata.section_id',
  'metadata.segment',
  'metadata.ship_mode'],
 'nota': 'Conteo exacto de puntos, no de documentos. La versión procede de una muestra; revisar todas las páginas para auditar el corpus.'}

## 6. Correspondencia del corpus remoto con el ACA 1

Se recorren las páginas del índice sin vectores. Se validan fuentes y las
16 secciones, hashes de cada fragmento, naturaleza simulada y versión común.
Cada intento usa un cliente nuevo sin filtro de versión y contrasta sus
puntos con el conteo total actual, incluso al repetir esta celda.
La cobertura compara el texto de los chunks con las secciones locales;
no confunde un fragmento con un documento completo. No modifica la nube.


In [25]:
import unicodedata
from dataclasses import replace
CORPUS_NUBE_OK = False
RECUPERACION_OK = DEMO_REAL_OK = False
CONFIG.update(corpus_cloud_verificado=False, llm_conexion_real_verificada=False,
              corpus_version_remota_verificada=None, filtro_version_remota_activado=False)
os.environ.pop("RAG_CORPUS_VERSION", None)
cfg = replace(cfg, corpus_version="") if cfg is not None else None
servicio = RagService(cfg) if cfg is not None else None
FRAGMENTOS_NUBE = []
auditoria_corpus = {"operacion": "corpus_cloud_vs_local", "correcto": False,
                   "corpus_sha256_local": corpus_sha, "errores": []}
def normalizar_comparacion(texto):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto)).strip()
if INDICE_OK:
    try:
        # Un filtro establecido en un intento anterior no puede ocultar
        # puntos ajenos o versiones mezcladas durante esta auditoría.
        cfg_auditoria, servicio_auditoria = cfg, servicio
        estado_auditoria = servicio_auditoria.estado_indice()
        if not estado_auditoria["compatible"] or not estado_auditoria["puntos"]:
            raise RagError("El índice actual no es compatible o está vacío; no se acredita correspondencia del corpus.")
        cursor, vistos = None, set()
        for pagina in range(1000):
            lote = servicio_auditoria.explorar_documentos(limit=100, offset=cursor)
            FRAGMENTOS_NUBE.extend(lote["fragmentos"])
            cursor = lote["next_page_offset"]
            if cursor is None:
                break
            if str(cursor) in vistos:
                raise RagError("Cursor repetido al auditar el índice; no se acredita cobertura completa.")
            vistos.add(str(cursor))
        else:
            raise RagError("Se alcanzó el límite de páginas de auditoría; no se acredita cobertura completa.")
        locales = {(x["fuente"], x["seccion"]): normalizar_comparacion(x["texto"]) for x in CATALOGO}
        cobertura = {key: set() for key in locales}
        encontrados, versiones = set(), set()
        for frag in FRAGMENTOS_NUBE:
            par = (frag["fuente"], frag["seccion"])
            meta = frag["metadatos"]
            if par not in locales:
                auditoria_corpus["errores"].append("Fuente/sección ajena al corpus local.")
                continue
            encontrados.add(par)
            versiones.add(meta.get("corpus_version", ""))
            if meta.get("content_sha256") != hashlib.sha256(frag["texto"].encode()).hexdigest():
                auditoria_corpus["errores"].append("Hash de fragmento no coincide: " + frag["seccion"])
            if meta.get("source_type") != "academico_simulado":
                auditoria_corpus["errores"].append("Naturaleza documental no coincide: " + frag["seccion"])
            texto = normalizar_comparacion(frag["texto"])
            inicio = locales[par].find(texto)
            if inicio < 0 or not texto:
                auditoria_corpus["errores"].append("Texto remoto no corresponde a sección local: " + frag["seccion"])
            else:
                cobertura[par].update(range(inicio, inicio + len(texto)))
        faltantes = sorted(set(locales) - encontrados)
        incompletas = [key[1] for key, texto in locales.items()
            if any(i not in cobertura[key] for i, char in enumerate(texto) if not char.isspace())]
        if faltantes or incompletas:
            auditoria_corpus["errores"].append("Hay secciones faltantes o sin cobertura textual completa.")
        if len(versiones) != 1 or not next(iter(versiones), "").startswith("superstore-academico-"):
            auditoria_corpus["errores"].append("La versión remota falta, está mezclada o no coincide con el esquema ACA 1.")
        ids = [str(f["id"]) for f in FRAGMENTOS_NUBE]
        if len(ids) != len(set(ids)):
            auditoria_corpus["errores"].append("Puntos repetidos en paginación.")
        if len(FRAGMENTOS_NUBE) != estado_auditoria["puntos"]:
            auditoria_corpus["errores"].append("El conteo paginado difiere del conteo de la colección.")
        CORPUS_NUBE_OK = not auditoria_corpus["errores"]
        auditoria_corpus.update(correcto=CORPUS_NUBE_OK, puntos_leidos=len(FRAGMENTOS_NUBE),
            secciones_encontradas=len(encontrados), secciones_sin_cobertura=incompletas,
            versiones_observadas=sorted(versiones), auditoria_sin_filtro_version=True,
            puntos_totales_coleccion=estado_auditoria["puntos"])
        if CORPUS_NUBE_OK:
            version_observada = next(iter(versiones))
            CONFIG["corpus_version_remota_verificada"] = version_observada
            ESTADO_INDICE = estado_auditoria
            # El filtro exige un índice de payload existente en modo estricto.
            # Auditar la colección completa no autoriza crear índices ni omitir errores.
            campos_indexados = estado_auditoria.get("metadatos_indexados", [])
            if any(campo in campos_indexados for campo in ("metadata.corpus_version", "corpus_version")):
                os.environ["RAG_CORPUS_VERSION"] = version_observada
                cfg = replace(cfg_auditoria, corpus_version=version_observada)
                servicio = RagService(cfg)
                CONFIG["filtro_version_remota_activado"] = True
            else:
                os.environ.pop("RAG_CORPUS_VERSION", None)
                cfg, servicio = cfg_auditoria, servicio_auditoria
                CONFIG["filtro_version_remota_activado"] = False
                print("Corpus completo auditado. No se añadió filtro de versión: no se observó su índice de payload. No se creó ningún índice.")
    except RagError as error:
        auditoria_corpus["errores"].append(str(error))
    except Exception as error:
        auditoria_corpus["errores"].append("Fallo controlado: " + type(error).__name__)
else:
    auditoria_corpus["errores"].append("Índice no disponible: no se sustituyó por un corpus local simulado.")
# También un fallo posterior a la lectura invalida el resultado completo.
CORPUS_NUBE_OK = not auditoria_corpus["errores"]
auditoria_corpus["correcto"] = CORPUS_NUBE_OK
if not CORPUS_NUBE_OK:
    CONFIG["corpus_version_remota_verificada"] = None
display(auditoria_corpus)
guardar_evidencia_nueva("corpus", auditoria_corpus)
CONFIG["corpus_cloud_verificado"] = CORPUS_NUBE_OK


{'operacion': 'corpus_cloud_vs_local',
 'correcto': True,
 'corpus_sha256_local': '1e34a469278740a59752a91c75d39308b48c7cf67e864d39923e2ae3a3f32ef2',
 'errores': [],
 'puntos_leidos': 47,
 'secciones_encontradas': 16,
 'secciones_sin_cobertura': [],
 'versiones_observadas': ['superstore-academico-575b82ca54a0'],
 'auditoria_sin_filtro_version': True,
 'puntos_totales_coleccion': 47}

## 7. Recuperación Top-k y demostración real

La primera comprobación recupera evidencia, sin LLM. Después, una sola
demostración utiliza el generador real únicamente si el índice y corpus
fueron verificados. No se copian las respuestas esperadas del banco.
Citas, contexto y tiempos proceden de la ejecución; los tiempos son del
pipeline y no incluyen renderizado del navegador.


In [26]:
RECUPERACION_OK = False
if CORPUS_NUBE_OK:
    try:
        resultado_busqueda = servicio.recuperar(CASOS[0]["turnos"][0], top_k=3)
        RECUPERACION_OK = any(f["seccion"] == "DEV_FURNITURE" for f in resultado_busqueda["fragmentos"])
        display(resultado_busqueda)
        guardar_evidencia_nueva("top_k", {"correcto": RECUPERACION_OK, **resultado_busqueda})
        if not RECUPERACION_OK:
            print("No apareció la evidencia mínima esperada. Revisar recuperación antes de evaluar el LLM.")
    except RagError as error:
        print("Recuperación no completada:", str(error))
    except Exception:
        print("Recuperación no completada. No se mostraron datos de conexión.")
else:
    print("Recuperación pendiente: primero debe publicarse y verificarse la base vectorial propia.")

# Auditoría del límite real del mismo tokenizer usado por el backend.
# No altera ni abrevia consultas para hacer pasar la prueba.
if servicio is not None and getattr(servicio, "_encoder", None) is not None:
    encoder_auditoria = servicio._encoder
    if hasattr(encoder_auditoria, "tokenizer"):
        longitudes = []
        for caso in CASOS:
            anterior = ""
            for numero, pregunta in enumerate(caso["turnos"], 1):
                consulta = (anterior[-1000:] + "\n" if anterior else "") + pregunta.strip()
                n = len(encoder_auditoria.tokenizer(consulta, truncation=False)["input_ids"])
                limite = encoder_auditoria.max_seq_length
                longitudes.append({"caso": caso["id"], "turno": numero, "tokens": n,
                                   "limite": limite, "compatible": n <= limite})
                anterior = pregunta
        display(longitudes)
        guardar_evidencia_nueva("longitudes_consulta", {"medicion_tokenizer_real": True, "filas": longitudes})
        if any(not x["compatible"] for x in longitudes):
            print("Hay consultas que el backend rechazará por longitud. Conservar la incidencia; no truncar ni inventar resultados.")

EJECUTAR_DEMO_REAL = True  # @param {type:"boolean"}
DEMO_REAL_OK = False
if EJECUTAR_DEMO_REAL and RECUPERACION_OK:
    try:
        DEMO = servicio.responder(CASOS[0]["turnos"][0], historial=[], top_k=3)
        display(DEMO)
        guardar_evidencia_nueva("demostracion_llm_real", DEMO)
        DEMO_REAL_OK = DEMO.get("modo") == "llm_rag_real" and bool(DEMO.get("llm_llamado"))
        print("Llamada real completada. La calidad sigue pendiente de revisión afirmación por afirmación.")
    except RagError as error:
        print("Demostración real no completada:", str(error))
    except Exception:
        print("Demostración real no completada. No se imprimieron detalles privados.")
else:
    print("No se realizó inferencia: demostración desactivada o recuperación sin verificar.")
CONFIG["llm_conexion_real_verificada"] = DEMO_REAL_OK
CONFIG["estado_integracion"] = (
    "Conexión RAG real demostrada; fidelidad, interfaz y cierre académico pendientes de sus evidencias."
    if DEMO_REAL_OK else "Implementación integrada; recorrido RAG real no completado en esta sesión."
)


{'fragmentos': [{'id': 23,
   'fuente': 'Manual_SLA_Logistica_Envios.md',
   'seccion': 'ENV_FIRST',
   'texto': 'Si se detecta una incidencia antes del despacho, se avisa de la nueva estimación sin ocultar el cambio bajo una promesa anterior. El seguimiento registra la hora del último evento disponible y la acción solicitada al operador ficticio. Si una persona necesita un compromiso real para una compra concreta, debe consultar al proveedor; el buscador académico solo recupera este manual.',
   'score': 0.58351326,
   'metadatos': {'doc_source': 'Manual_SLA_Logistica_Envios.md',
    'section_id': 'ENV_FIRST',
    'page_section': 'ENV_FIRST | Envío prioritario: First Class',
    'category': 'General',
    'ship_mode': 'First Class',
    'segment': 'All',
    'region': 'All',
    'sub_categories': [],
    'source_type': 'academico_simulado',
    'corpus_version': 'superstore-academico-575b82ca54a0',
    'content_sha256': 'eaec14ce06f87a7b73a54268725669bcc7b71fe97a7351d6b651f0c6a3d9269e

[{'caso': 'C01', 'turno': 1, 'tokens': 37, 'limite': 128, 'compatible': True},
 {'caso': 'C02', 'turno': 1, 'tokens': 38, 'limite': 128, 'compatible': True},
 {'caso': 'C03', 'turno': 1, 'tokens': 30, 'limite': 128, 'compatible': True},
 {'caso': 'C04', 'turno': 1, 'tokens': 32, 'limite': 128, 'compatible': True},
 {'caso': 'C05', 'turno': 1, 'tokens': 48, 'limite': 128, 'compatible': True},
 {'caso': 'C06', 'turno': 1, 'tokens': 34, 'limite': 128, 'compatible': True},
 {'caso': 'C07', 'turno': 1, 'tokens': 21, 'limite': 128, 'compatible': True},
 {'caso': 'C07', 'turno': 2, 'tokens': 37, 'limite': 128, 'compatible': True},
 {'caso': 'C08', 'turno': 1, 'tokens': 20, 'limite': 128, 'compatible': True},
 {'caso': 'C09', 'turno': 1, 'tokens': 22, 'limite': 128, 'compatible': True},
 {'caso': 'C10', 'turno': 1, 'tokens': 34, 'limite': 128, 'compatible': True},
 {'caso': 'C11', 'turno': 1, 'tokens': 55, 'limite': 128, 'compatible': True},
 {'caso': 'C12', 'turno': 1, 'tokens': 12, 'limite':

{'modo': 'llm_rag_real',
 'modelo': 'google/gemini-3.8-flash',
 'modelo_version_reportada': None,
 'modelo_identificacion': 'Identificador solicitado al SDK; versión resuelta no expuesta',
 'respuesta': 'En este escenario académico simulado:\n\n- Plazo y reporte: El reporte debe presentarse dentro de los cinco días de ejemplo posteriores a la recepción, indicando la referencia de pedido, el producto y la cantidad afectada [F2].\n- Evidencias: Se deben conservar el empaque y las piezas, y enviar fotografías del producto completo, del daño y de la etiqueta del paquete [F2].\n- Coordinación de la recogida: No cuento con esa información en los documentos disponibles.',
 'fuentes': [{'etiqueta': 'F2',
   'fuente': 'Politica_Devoluciones_Garantias_Superstore.md',
   'seccion': 'DEV_FURNITURE'}],
 'fragmentos': [{'id': 23,
   'fuente': 'Manual_SLA_Logistica_Envios.md',
   'seccion': 'ENV_FIRST',
   'texto': 'Si se detecta una incidencia antes del despacho, se avisa de la nueva estimación sin 

Llamada real completada. La calidad sigue pendiente de revisión afirmación por afirmación.


## 8. Aplicación Streamlit integrada

La app ofrece **Chat con fuentes**, **Documentos y fragmentos** y
**Supervisión del índice**. El historial
es por sesión; se muestran únicamente fuentes citadas efectivamente y los
fragmentos recuperados se identifican por separado. Los filtros conservan
secciones generales aplicables. El panel no modifica documentos remotos.

El servidor se inicia solo en el puerto 8501 y no se finalizan procesos de
terceros. La visualización utiliza el proxy autenticado oficial de Colab,
no ngrok ni un túnel público. La sesión debe permanecer abierta. Si Qdrant
falla, la app conserva el error controlado, sin presentar respuestas ficticias.
Una página visible no acredita que el chatbot esté operativo de extremo a extremo.
Si cambia secretos, modelo o filtros tras iniciar la app, active
`REINICIAR_INTERFAZ_PROPIA` y ejecute de nuevo esta celda. Solo reinicia el
proceso identificado que creó esta misma celda, nunca otro servidor.


In [27]:
import socket, tempfile, time
INICIAR_INTERFAZ = True  # @param {type:"boolean"}
REINICIAR_INTERFAZ_PROPIA = False  # @param {type:"boolean"}
PUERTO_APP = 8501
def puerto_ocupado():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", PUERTO_APP)) == 0

if INICIAR_INTERFAZ:
    anterior = globals().get("PROCESO_STREAMLIT_PROPIO")
    propio_activo = anterior is not None and anterior.poll() is None
    if propio_activo and REINICIAR_INTERFAZ_PROPIA:
        anterior.terminate()
        try:
            anterior.wait(timeout=5)
            propio_activo = False
        except subprocess.TimeoutExpired:
            print("El proceso propio no terminó en 5 segundos. Se conservó sin forzar su cierre.")
    elif propio_activo:
        print("Se conserva el servidor propio. Si cambió configuración, active REINICIAR_INTERFAZ_PROPIA para que herede los valores nuevos.")
    if puerto_ocupado() and not propio_activo:
        print("Puerto 8501 ocupado por un proceso no iniciado por esta celda. No se detuvo ni reemplazó.")
    else:
        if not propio_activo:
            # El log temporal está fuera del paquete y no se muestra ni exporta.
            LOG_PRIVADO_STREAMLIT = tempfile.TemporaryFile(mode="w+b")
            PROCESO_STREAMLIT_PROPIO = subprocess.Popen(
                [sys.executable, "-m", "streamlit", "run", str(BASE / "app.py"),
                 "--server.port=8501", "--server.address=127.0.0.1",
                 "--server.headless=true", "--browser.gatherUsageStats=false"],
                cwd=BASE, stdout=LOG_PRIVADO_STREAMLIT, stderr=subprocess.STDOUT,
                env=os.environ.copy(),
            )
        for _ in range(20):
            if puerto_ocupado() or PROCESO_STREAMLIT_PROPIO.poll() is not None:
                break
            time.sleep(1)
        if puerto_ocupado() and PROCESO_STREAMLIT_PROPIO.poll() is None:
            print("Servidor local iniciado. La verificación visual y RAG se registran por separado.")
            try:
                from google.colab import output
                output.serve_kernel_port_as_iframe(PUERTO_APP, height=850, cache_in_notebook=False)
            except Exception:
                print("No se pudo abrir el visor autenticado. Fuera de Colab, abra http://127.0.0.1:8501 en el mismo equipo.")
        else:
            print("La aplicación no quedó disponible en el plazo de comprobación. Revisar el entorno de forma privada.")


Se conserva el servidor propio. Si cambió configuración, active REINICIAR_INTERFAZ_PROPIA para que herede los valores nuevos.
Servidor local iniciado. La verificación visual y RAG se registran por separado.


<IPython.core.display.Javascript object>

## 9. Pruebas de software sin servicios externos

Estos tests utilizan dobles sintéticos explícitos. Comprueban seguridad,
contrato, citas, errores, interfaz y evaluación. **No son respuestas reales
del chatbot ni acreditan disponibilidad de Qdrant o fidelidad del LLM.**
La cantidad se obtiene de la ejecución, no de una cifra escrita de antemano.


In [28]:
import contextlib, unittest
suite = unittest.defaultTestLoader.discover(str(BASE), pattern="test_*.py", top_level_dir=str(BASE))
buffer_tests = io.StringIO()
with contextlib.redirect_stdout(buffer_tests), contextlib.redirect_stderr(buffer_tests):
    resultado_tests = unittest.TextTestRunner(stream=buffer_tests, verbosity=1).run(suite)
resumen_tests = {"proposito": "pruebas_unitarias_sinteticas_no_evidencia_llm",
    "ejecutadas": resultado_tests.testsRun, "errores": len(resultado_tests.errors),
    "fallos": len(resultado_tests.failures), "omitidas": len(resultado_tests.skipped),
    "correcto": resultado_tests.wasSuccessful(),
    "tests_con_fallo": [test.id() for test, _ in resultado_tests.errors + resultado_tests.failures]}
display(resumen_tests)
guardar_evidencia_nueva("tests_offline", resumen_tests)
del buffer_tests


{'proposito': 'pruebas_unitarias_sinteticas_no_evidencia_llm',
 'ejecutadas': 145,
 'errores': 0,
 'fallos': 0,
 'omitidas': 0,
 'correcto': True,
 'tests_con_fallo': []}

## 10. Protocolo de evaluación y registro sin sobrescritura

El banco tiene 12 casos y 13 turnos: consultas directas, paráfrasis, una
compleja multifuente, seguimiento, ausencia de información, resistencia a
instrucciones y ambigüedad. C05 utiliza Top-k=5; los otros, Top-k=3.
Cada caso inicia historial propio; solo C07 conserva sus dos turnos.

Se miden latencia hasta respuesta completa, respaldo de afirmaciones,
pertinencia de citas, completitud y abstención. Las cuatro pruebas
funcionales requieren observar la UI y dejar evidencia. Un denominador
cero produce `null`/pendiente, no una calidad inventada de 0 % o 100 %.

La configuración se congela por corrida. Los casos ejecutados, incluso
fallidos, no se reemplazan: para repetir después de corregir hay que crear
otra corrida. Cada caso termina con un checkpoint; interrupciones conservan
la evidencia ya obtenida. Los resultados esperados sirven para revisión,
nunca para generar respuestas en ausencia del servicio.


In [29]:
display([{ "id": c["id"], "tipo": c["tipo"], "turnos": len(c["turnos"]),
           "top_k": c["top_k"], "secciones_esperadas": c["secciones_esperadas"] } for c in CASOS])
display(FUNCIONALES)
CORRIDA = "corrida_integrada_01"  # @param {type:"string"}
if not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", CORRIDA):
    raise ValueError("Nombre de corrida no válido.")
RUTA_REGISTRO = BASE / "datos" / (CORRIDA + ".json")
REGISTRO = inicializar_registro(RUTA_REGISTRO, CASOS, FUNCIONALES, APP_VERSION)
print("Corrida conservada:", CORRIDA)
display([{ "id": r["id"], "estado": r["estado"] } for r in REGISTRO["casos"]])


[{'id': 'C01',
  'tipo': 'directa_devolucion_mueble',
  'turnos': 1,
  'top_k': 3,
  'secciones_esperadas': ['DEV_FURNITURE']},
 {'id': 'C02',
  'tipo': 'directa_logistica_estandar',
  'turnos': 1,
  'top_k': 3,
  'secciones_esperadas': ['ENV_STANDARD']},
 {'id': 'C03',
  'tipo': 'directa_soporte_copiadora',
  'turnos': 1,
  'top_k': 3,
  'secciones_esperadas': ['SOP_COPIERS']},
 {'id': 'C04',
  'tipo': 'directa_devolucion_corporativa',
  'turnos': 1,
  'top_k': 3,
  'secciones_esperadas': ['DEV_CORPORATE']},
 {'id': 'C05',
  'tipo': 'compleja_multifuente_garantia_y_soporte',
  'turnos': 1,
  'top_k': 5,
  'secciones_esperadas': ['SOP_PHONES', 'DEV_TECH']},
 {'id': 'C06',
  'tipo': 'parafrasis_entrega_mismo_dia',
  'turnos': 1,
  'top_k': 3,
  'secciones_esperadas': ['ENV_SAME_DAY']},
 {'id': 'C07',
  'tipo': 'conversacion_dos_turnos',
  'turnos': 2,
  'top_k': 3,
  'secciones_esperadas': ['ENV_FIRST']},
 {'id': 'C08',
  'tipo': 'fuera_de_dominio',
  'turnos': 1,
  'top_k': 3,
  'secci

[{'id': 'F01',
  'tipo': 'panel_documentos_e_indice',
  'pasos': ['Abrir el panel de documentos y verificar que enumera los tres Markdown del corpus académico.',
   'Comprobar nombre, versión o huella disponible, estado de carga y aviso de simulación.',
   'Consultar el estado del índice sin insertar ni eliminar datos durante esta prueba.',
   'Comparar el conteo y los identificadores informados con el manifiesto de la versión cargada.',
   'Ejecutar C01 y comprobar que la fuente y sección mostradas existen en los documentos indexados.',
   'Si el índice remoto no está conectado, revisar que el panel diga pendiente o no disponible, no operativo.'],
  'resultado_esperado': 'El panel diferencia archivos disponibles, fragmentos preparados e índice realmente verificado. Muestra las tres fuentes correctas sin confundir el CSV estructurado con un cuarto manual. Los conteos provienen del manifiesto o del servicio consultado y no son valores decorativos. Se informa el origen local o remoto de 

Corrida conservada: corrida_integrada_01


[{'id': 'C01', 'estado': 'no_ejecutado'},
 {'id': 'C02', 'estado': 'no_ejecutado'},
 {'id': 'C03', 'estado': 'no_ejecutado'},
 {'id': 'C04', 'estado': 'no_ejecutado'},
 {'id': 'C05', 'estado': 'no_ejecutado'},
 {'id': 'C06', 'estado': 'no_ejecutado'},
 {'id': 'C07', 'estado': 'no_ejecutado'},
 {'id': 'C08', 'estado': 'no_ejecutado'},
 {'id': 'C09', 'estado': 'no_ejecutado'},
 {'id': 'C10', 'estado': 'no_ejecutado'},
 {'id': 'C11', 'estado': 'no_ejecutado'},
 {'id': 'C12', 'estado': 'no_ejecutado'}]

In [34]:
# Activar solo cuando esté lista la conexión real. Puede consumir cuota.
EJECUTAR_BATERIA_REAL = True  # @param {type:"boolean"}
if EJECUTAR_BATERIA_REAL and RECUPERACION_OK and DEMO_REAL_OK and resumen_tests["correcto"]:
    for caso in CASOS:
        existente = next(r for r in REGISTRO["casos"] if r["id"] == caso["id"])
        if existente["estado"] != "no_ejecutado":
            print(caso["id"], "conservado:", existente["estado"])
            continue
        try:
            REGISTRO = ejecutar_bateria(servicio.responder, CASOS, REGISTRO, CATALOGO,
                                       CONFIG, confirmar_real=True, ids=[caso["id"]])
            # Copia única por checkpoint y registro de la corrida para reanudar.
            guardar_evidencia_nueva(CORRIDA + "_" + caso["id"], REGISTRO)
            guardar_json(RUTA_REGISTRO, REGISTRO)
            fila = next(r for r in REGISTRO["casos"] if r["id"] == caso["id"])
            print(caso["id"], fila["estado"])
        except Exception as error:
            print("Batería detenida sin reemplazar resultados:", type(error).__name__)
            break
elif EJECUTAR_BATERIA_REAL:
    print("Batería bloqueada: falta verificar el recorrido índice → contexto → LLM o resolver los tests de software.")
else:
    print("No se llamó al LLM. Active EJECUTAR_BATERIA_REAL para evaluar cuando esté disponible el sistema.")
display(calcular_metricas(REGISTRO, CASOS))


C01 ejecutado_sin_revision
C02 ejecutado_sin_revision
C03 ejecutado_sin_revision
C04 ejecutado_sin_revision
C05 ejecutado_sin_revision
C06 ejecutado_sin_revision
C07 ejecutado_sin_revision
C08 ejecutado_sin_revision
C09 ejecutado_sin_revision
C10 ejecutado_sin_revision
C11 ejecutado_sin_revision
C12 ejecutado_sin_revision


{'casos_previstos': 12,
 'casos_ejecutados_completos': 12,
 'casos_revisados': 0,
 'casos_con_error': 0,
 'afirmaciones_revisadas': 0,
 'afirmaciones_respaldadas': 0,
 'fidelidad_pct': None,
 'citas_revisadas': 0,
 'citas_correctas': 0,
 'consultas_con_evidencia_revisadas': 0,
 'respuestas_sustentadas': 0,
 'respuestas_sustentadas_pct': None,
 'citas_correctas_pct': None,
 'casos_abstencion_previstos': 3,
 'casos_abstencion_revisados': 0,
 'abstencion_correcta_pct': None,
 'casos_que_cumplen': 0,
 'cumplimiento_casos_pct': None,
 'turnos_con_latencia': 13,
 'latencia_media_s': 7.2192,
 'latencia_mediana_s': 7.1146,
 'latencia_p95_s': 9.985,
 'funcionales_revisadas': 0,
 'funcionales_aprobadas': 0,
 'nota': 'None significa pendiente/no calculable, nunca cero de rendimiento. Calidad mediante revisión documentada de afirmaciones y citas, no coincidencia de palabras. Identificar revisión asistida en revisor/observaciones; la confirmación humana del equipo es distinta y no se presume. Muest

In [35]:
# Auditoría de respuestas generadas con Qdrant + Gemini (sin credenciales)
import json
resumen_auditoria = {
    "casos": [
        {"id": fila["id"], "estado": fila["estado"],
         "turnos": [{"pregunta": t["pregunta"], "respuesta": t["respuesta"],
                     "fuentes": t["fuentes"], "fragmentos": t["fragmentos"],
                     "latencia_s": t["latencia_s"]} for t in fila["turnos"]]}
        for fila in REGISTRO["casos"]
    ],
    "metricas": calcular_metricas(REGISTRO, CASOS),
}
print("ACA2_AUDITORIA_JSON=" + json.dumps(resumen_auditoria, ensure_ascii=False))

ACA2_AUDITORIA_JSON={"casos": [{"id": "C01", "estado": "ejecutado_sin_revision", "turnos": [{"pregunta": "La silla de mi pedido llegó rota. Según el escenario académico, ¿en qué plazo debo reportarla, qué evidencias envío y cuándo se coordina la recogida?", "respuesta": "En este escenario académico simulado (cuyas políticas y plazos no representan condiciones comerciales reales), el reporte debe presentarse dentro de los cinco días de ejemplo posteriores a la recepción [F2]. Las evidencias requeridas consisten en conservar el empaque y las piezas, tomar fotografías del producto completo, del daño y de la etiqueta del paquete [F2], e indicar la referencia de pedido, producto y cantidad afectada [F2]. Respecto a cuándo se coordina la recogida, no cuento con esa información en los documentos disponibles.", "fuentes": [{"fuente": "Politica_Devoluciones_Garantias_Superstore.md", "seccion": "DEV_FURNITURE"}], "fragmentos": [{"fuente": "Manual_SLA_Logistica_Envios.md", "seccion": "ENV_FIRST",

In [36]:
# Diagnóstico sin LLM: cobertura de secciones
for cid in ("C02", "C06"):
    caso = next(c for c in CASOS if c["id"] == cid)
    for k in (3, 5, 8):
        try:
            rec = servicio.recuperar(caso["turnos"][0], top_k=k)
            print(json.dumps({"caso":cid, "top_k":k, "secciones":[f["seccion"] for f in rec["fragmentos"]]}, ensure_ascii=False))
        except RagError as exc:
            print(json.dumps({"caso":cid, "top_k":k, "incidencia":str(exc)}, ensure_ascii=False))

{"caso": "C02", "top_k": 3, "secciones": ["ENV_DATA_SCOPE", "ENV_STANDARD", "ENV_FIRST"]}
{"caso": "C02", "top_k": 5, "secciones": ["ENV_DATA_SCOPE", "ENV_STANDARD", "ENV_FIRST", "ENV_STANDARD", "ENV_SAME_DAY"]}
{"caso": "C02", "top_k": 8, "secciones": ["ENV_DATA_SCOPE", "ENV_STANDARD", "ENV_FIRST", "ENV_STANDARD", "ENV_SAME_DAY", "ENV_STANDARD", "ENV_DELAY", "ENV_SECOND"]}
{"caso": "C06", "top_k": 3, "secciones": ["ENV_STANDARD", "ENV_FIRST", "ENV_SECOND"]}
{"caso": "C06", "top_k": 5, "secciones": ["ENV_STANDARD", "ENV_FIRST", "ENV_SECOND", "ENV_SAME_DAY", "ENV_FIRST"]}
{"caso": "C06", "top_k": 8, "secciones": ["ENV_STANDARD", "ENV_FIRST", "ENV_SECOND", "ENV_SAME_DAY", "ENV_FIRST", "DEV_OFFICE", "DEV_CORPORATE", "DEV_TRACE"]}


# Reensayo aislado: ampliar cobertura a top-5 sin modificar la primera corrida
import copy, json
if not (INDICE_OK and CORPUS_NUBE_OK and DEMO_REAL_OK):
    raise RuntimeError('Dependencias reales sin validar')
CASOS_REENSAYO = copy.deepcopy(CASOS)
for caso in CASOS_REENSAYO:
    if caso['id'] in ('C02', 'C06'):
        caso['top_k'] = 5
CONFIG_REENSAYO = {**CONFIG, 'usar_top_k_por_caso': True}
RUTA_REENSAYO = BASE / 'datos' / 'corrida_integrada_02_top5.json'
REGISTRO_REENSAYO = inicializar_registro(RUTA_REENSAYO, CASOS_REENSAYO, FUNCIONALES, APP_VERSION)
for cid in ('C02', 'C06'):
    previo = next(f for f in REGISTRO_REENSAYO['casos'] if f['id'] == cid)
    if previo['estado'] == 'no_ejecutado':
        REGISTRO_REENSAYO = ejecutar_bateria(servicio.responder, CASOS_REENSAYO,
            REGISTRO_REENSAYO, CATALOGO, CONFIG_REENSAYO, confirmar_real=True, ids=[cid])
        guardar_json(RUTA_REENSAYO, REGISTRO_REENSAYO)
for cid in ('C02', 'C06'):
    fila = next(f for f in REGISTRO_REENSAYO['casos'] if f['id'] == cid)
    print('ACA2_REENSAYO_JSON=' + json.dumps({'id':cid, 'estado':fila['estado'],
        'top_k':fila.get('top_k'), 'turnos':[{'respuesta':t['respuesta'],
        'fuentes':t['fuentes'], 'fragmentos':t['fragmentos']} for t in fila['turnos']]}, ensure_ascii=False))

In [38]:
# Reensayo aislado top-5: conserva la corrida de 12 casos
import copy, json
assert INDICE_OK and CORPUS_NUBE_OK and DEMO_REAL_OK
CASOS_REENSAYO = copy.deepcopy(CASOS)
for caso in CASOS_REENSAYO:
    if caso['id'] in ('C02', 'C06'):
        caso['top_k'] = 5
CONFIG_REENSAYO = {**CONFIG, 'usar_top_k_por_caso': True}
RUTA_REENSAYO = BASE / 'datos' / 'corrida_integrada_02_top5.json'
REGISTRO_REENSAYO = inicializar_registro(RUTA_REENSAYO, CASOS_REENSAYO, FUNCIONALES, APP_VERSION)
for cid in ('C02', 'C06'):
    fila = next(f for f in REGISTRO_REENSAYO['casos'] if f['id'] == cid)
    if fila['estado'] == 'no_ejecutado':
        REGISTRO_REENSAYO = ejecutar_bateria(servicio.responder, CASOS_REENSAYO,
            REGISTRO_REENSAYO, CATALOGO, CONFIG_REENSAYO, confirmar_real=True, ids=[cid])
        guardar_json(RUTA_REENSAYO, REGISTRO_REENSAYO)
for cid in ('C02', 'C06'):
    fila = next(f for f in REGISTRO_REENSAYO['casos'] if f['id'] == cid)
    print('ACA2_REENSAYO_JSON=' + json.dumps({'id':cid, 'estado':fila['estado'],
        'top_k':fila.get('top_k'), 'turnos':[{'respuesta':t['respuesta'],
        'fuentes':t['fuentes'], 'fragmentos':t['fragmentos']} for t in fila['turnos']]}, ensure_ascii=False))

ACA2_REENSAYO_JSON={"id": "C02", "estado": "ejecutado_sin_revision", "top_k": 5, "turnos": [{"respuesta": "En el marco de este escenario académico simulado, para Standard Class el manual establece una estimación ficticia de cuatro a seis días hábiles que se cuentan después de confirmar el despacho, y no desde la creación del pedido [F4]. Por otro lado, la resta entre Order Date y Ship Date no demuestra la entrega al cliente; dicha diferencia únicamente describe un intervalo de preparación en el dataset y no mide la duración del transporte ni la fecha de recepción [F1], dado que estos valores corresponden al pedido y al despacho, no a la recepción por parte del cliente [F2].", "fuentes": [{"fuente": "Manual_SLA_Logistica_Envios.md", "seccion": "ENV_STANDARD"}, {"fuente": "Manual_SLA_Logistica_Envios.md", "seccion": "ENV_DATA_SCOPE"}, {"fuente": "Manual_SLA_Logistica_Envios.md", "seccion": "ENV_STANDARD"}], "fragmentos": [{"fuente": "Manual_SLA_Logistica_Envios.md", "seccion": "ENV_DATA_

## 11. Revisión trazable y evidencias de interfaz

Leer la respuesta, sus fragmentos y cada atribución. Registrar quién revisó
y dónde está la evidencia. Si revisa un asistente, identificar la revisión
como asistida; la confirmación del equipo es otro paso. No aprobar una
respuesta solo porque se generó o porque el backend aceptó sus citas.

`REVISIONES` admite diccionarios con: caso_id, revisor, evidencia,
afirmaciones_totales, afirmaciones_respaldadas, citas_totales,
citas_correctas, cumple, abstencion_correcta y observaciones.
`CONTROLES_UI` admite: prueba_id, cumple, evidencia, revisor, observaciones.
Mantener listas vacías mientras no exista evidencia. Capturar chat complejo,
seguimiento, abstención y panel documental desde la aplicación real.


In [31]:
REVISIONES = []  # Completar solo a partir de observaciones documentadas.
CONTROLES_UI = []
for revision in REVISIONES:
    REGISTRO = revisar_caso(REGISTRO, CASOS, **revision)
    guardar_evidencia_nueva(CORRIDA + "_revision", REGISTRO)
    guardar_json(RUTA_REGISTRO, REGISTRO)
for control in CONTROLES_UI:
    REGISTRO = registrar_funcional(REGISTRO, **control)
    guardar_evidencia_nueva(CORRIDA + "_control_ui", REGISTRO)
    guardar_json(RUTA_REGISTRO, REGISTRO)
display(calcular_metricas(REGISTRO, CASOS))


{'casos_previstos': 12,
 'casos_ejecutados_completos': 0,
 'casos_revisados': 0,
 'casos_con_error': 0,
 'afirmaciones_revisadas': 0,
 'afirmaciones_respaldadas': 0,
 'fidelidad_pct': None,
 'citas_revisadas': 0,
 'citas_correctas': 0,
 'consultas_con_evidencia_revisadas': 0,
 'respuestas_sustentadas': 0,
 'respuestas_sustentadas_pct': None,
 'citas_correctas_pct': None,
 'casos_abstencion_previstos': 3,
 'casos_abstencion_revisados': 0,
 'abstencion_correcta_pct': None,
 'casos_que_cumplen': 0,
 'cumplimiento_casos_pct': None,
 'turnos_con_latencia': 0,
 'latencia_media_s': None,
 'latencia_mediana_s': None,
 'latencia_p95_s': None,
 'funcionales_revisadas': 0,
 'funcionales_aprobadas': 0,
 'nota': 'None significa pendiente/no calculable, nunca cero de rendimiento. Calidad mediante revisión documentada de afirmaciones y citas, no coincidencia de palabras. Identificar revisión asistida en revisor/observaciones; la confirmación humana del equipo es distinta y no se presume. Muestra pequ

## 12. Exportar métricas y materiales de entrega

Los registros, tabla CSV, incidencias, póster y documentación se generan
desde esta corrida. Las métricas no medidas quedan pendientes. El guion
plantea un video de 4:30 dentro del rango exigido de 3–5 minutos; el video
debe grabarse con voz real y demostrar el sistema que efectivamente funciona.

Cada exportación usa una carpeta nueva para conservar las anteriores. No
se marcan autores confirmados, enlaces accesibles, rotación de claves,
capturas o video como completados sin verificarlos. Incluir la app y código
en el repositorio solicitado por el profesor, sin secretos. El proxy de
Colab es temporal y autenticado: no equivale a publicar una app permanente.


In [32]:
import shutil
from materiales_aca2 import generar_todo, empaquetar
EXPORTAR_MATERIALES = True  # @param {type:"boolean"}
RUTA_ZIP_FINAL = None
if EXPORTAR_MATERIALES:
    validar_coherencia_exportacion(REGISTRO, CONFIG)
    stamp_exportacion = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    ENTREGA = BASE.parent / ("ACA2_Entrega_" + stamp_exportacion)
    # Carpeta nueva: no reemplaza entregas anteriores ni la configuración de trabajo.
    def excluir_archivos_privados(carpeta, nombres):
        descartados = set(shutil.ignore_patterns("__pycache__", "*.pyc", "*.log", ".env", ".env.*", "secrets.toml", ".git", ".venv")(carpeta, nombres))
        descartados.discard(".env.example")
        return descartados
    shutil.copytree(BASE, ENTREGA, ignore=excluir_archivos_privados)
    guardar_json(ENTREGA / "config_proyecto.json", CONFIG)
    exportar_registros(REGISTRO, CASOS, ENTREGA / "resultados")
    materiales = generar_todo(ENTREGA, CONFIG, REGISTRO, CORPUS)
    RUTA_ZIP_FINAL = empaquetar(ENTREGA, ENTREGA.with_suffix(".zip"))
    print("Exportación preservada:", ENTREGA.name)
    print("Pendientes reales:", len(materiales["pendientes"]))
    display(FileLink(str(RUTA_ZIP_FINAL)))
    display(FileLink(materiales["poster"]))
DESCARGAR_ZIP = False  # @param {type:"boolean"}
if DESCARGAR_ZIP and RUTA_ZIP_FINAL:
    try:
        from google.colab import files
        files.download(str(RUTA_ZIP_FINAL))
    except Exception:
        print("Descargue el ZIP desde el panel Archivos; la exportación permanece guardada.")


Exportación preservada: ACA2_Entrega_20261115T215501727971Z
Pendientes reales: 11


/content/ACA2_Entrega_20261115T215501727971Z.zip

/content/ACA2_Entrega_20261115T215501727971Z/materiales/Poster_ACA2_BORRADOR.pdf

## 13. Lista de cierre y limitaciones

El 15 de noviembre quedó publicada y verificada la colección propia en
Qdrant Cloud; el registro está embebido en este paquete. Para cerrar,
repetir la comprobación desde este cuaderno, ejecutar y revisar la batería,
validar los cuatro controles de
UI; incorporar capturas reales y conclusiones basadas en resultados; confirmar
autores; comprobar repositorio, notebook, app y video. Una implementación
no se declara validada mientras estas evidencias estén pendientes.

No se rotan ni se borran claves automáticamente. Que el corpus sea simulado
no hace inocua una credencial: conserva acceso a un servicio real. Una clave
que ya figuraba en el original debe revisarse/rotarse por su propietario;
quitarla de esta copia no elimina versiones históricas.

Fuentes técnicas: guía oficial del ACA 2 y distribución incluidas en
`referencia/guias`; código saneado del ACA 1 en `referencia`; documentación
oficial del SDK y visualización de Colab:
[Colab](https://colab.research.google.com/notebooks/welcome.ipynb),
[Colab Pro](https://colab.research.google.com/notebooks/pro.ipynb),
[implementación de la visualización autenticada](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py).
La función de ventana está deprecada; este notebook usa iframe oficial.


In [33]:
PENDIENTES = pendientes_cierre(REGISTRO, CONFIG, CASOS)
print("Estado técnico observado en esta sesión:")
display({"indice_real_verificado": INDICE_OK, "corpus_remoto_verificado": CORPUS_NUBE_OK,
         "recuperacion_verificada": RECUPERACION_OK, "demo_llm_real": DEMO_REAL_OK,
         "nota": "La fidelidad se evalúa aparte; no se deduce de estas banderas."})
print("Pendientes para cierre académico:")
for item in PENDIENTES:
    print("-", item)
if not (INDICE_OK and CORPUS_NUBE_OK and RECUPERACION_OK and DEMO_REAL_OK):
    print("ENTREGA NO VALIDADA END-TO-END: se conservó el bloqueo real; no se inventaron resultados.")
print("El ACA 1 y la colección del compañero se conservan. La escritura propia, si se autorizó, tiene un registro separado en 4B.")


Estado técnico observado en esta sesión:


{'indice_real_verificado': True,
 'corpus_remoto_verificado': True,
 'recuperacion_verificada': True,
 'demo_llm_real': True,
 'nota': 'La fidelidad se evalúa aparte; no se deduce de estas banderas.'}

Pendientes para cierre académico:
- Conectividad Qdrant: publicar y verificar la colección propia en Qdrant Cloud; la preparación local de corpus y embeddings no acredita publicación cloud.
- Evaluación: ejecutar y revisar los 12 casos con recuperación cloud y LLM reales; una prueba aislada del LLM no acredita RAG.
- Evaluación: documentar las 4 pruebas funcionales con evidencias reales; distinguir las comprobaciones con fixtures.
- Publicación: completar dato real url_app
- Publicación: completar dato real url_repositorio
- Publicación: completar dato real url_video
- Materiales: incorporar captura real sin secretos captura_chat
- Materiales: incorporar captura real sin secretos captura_documentos
- Publicación: probar los enlaces con los permisos requeridos para el profesor; no asumir acceso anónimo.
- Video: grabar y verificar 3 a 5 minutos con voz real y pantalla; el guion no sustituye el archivo audiovisual.
- Seguridad: rotación/revocación de credenciales previamente expuestas no